### Source Tables:
- `_exponent._bronze_srgry_dmart.rpt_case_mtrl_use_vw` — material/device usage in surgical cases
- `_exponent._bronze_srgry_dmart.rpt_mtrl_vw` — material/device reference data
- `_exponent._bronze_srgry_dmart.rpt_case_vw` — surgical case information (contains VST_ID_CD for visit linkage)

### Strategy:
- Extract surgical materials/devices used during cases
- Link to SCM visits via `rpt_case_vw.VST_ID_CD` = `cv3clientvisit.VisitIDCode`
- Link to patients via `cv3clientvisit.ClientGUID`
- device_concept_id = 0 (no standard vocabulary codes available - proprietary material names)
- device_source_value = material name
- device_type_concept_id = 32817 (EHR)

### Notes:
- Materials do not have HCPCS/SNOMED codes, so standard concept mapping is not possible
- Dates come from case start/end timestamps
- Quantity from MTRL_QTY_NUM
- Cancelled cases (CNCLD_IND = 1) are excluded

In [ ]:
%sql
-- Exploration: Verify the visit linkage via VisitIDCode
-- SELECT 
--   c.VST_ID_CD,
--   v.VisitIDCode,
--   v.GUID AS visit_guid,
--   v.ClientGUID,
--   m.MTRL_NM,
--   mu.MTRL_QTY_NUM
-- FROM _exponent._bronze_srgry_dmart.rpt_case_mtrl_use_vw mu
-- INNER JOIN _exponent._bronze_srgry_dmart.rpt_mtrl_vw m ON m.MTRL_DIM_ID = mu.MTRL_DIM_ID
-- INNER JOIN _exponent._bronze_srgry_dmart.rpt_case_vw c ON c.CASE_FCT_ID = mu.CASE_FCT_ID
-- LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit v
--   ON CAST(c.VST_ID_CD AS STRING) = CAST(v.VisitIDCode AS STRING)
-- WHERE c.CNCLD_IND = 0
-- LIMIT 20;

In [ ]:
%sql
-- Create silver view for SCM device exposure from surgery data mart
CREATE OR REPLACE TEMPORARY VIEW silver_device_exposure_scm AS
SELECT
  -- Unique source key
  CONCAT_WS(
    CHR(31),
    'allscripts_scm',
    'rpt_case_mtrl_use_vw',
    'MTRL_USE_FCT_ID',
    CAST(mu.MTRL_USE_FCT_ID AS STRING)
  ) AS device_exposure_source_value,

  -- Person ID via source_to_person
  stp.person_id AS person_id,

  -- Device concept (0 since no standard codes available)
  CAST(0 AS INT) AS device_concept_id,

  -- Dates from case timestamps
  CAST(mu.CASE_STRT_TS AS DATE) AS device_exposure_start_date,
  mu.CASE_STRT_TS AS device_exposure_start_datetime,
  CAST(COALESCE(mu.CASE_END_TS, mu.CASE_STRT_TS) AS DATE) AS device_exposure_end_date,
  COALESCE(mu.CASE_END_TS, mu.CASE_STRT_TS) AS device_exposure_end_datetime,

  -- Type concept: EHR
  CAST(32817 AS INT) AS device_type_concept_id,

  -- Unique device ID (manufacturer part number if available)
  COALESCE(m.MFC_PART_NUM, m.MFC_MODL_NUM) AS unique_device_id,

  -- Quantity
  CAST(COALESCE(mu.MTRL_QTY_NUM, 1) AS DOUBLE) AS quantity,

  -- Provider ID (surgeon) - TODO: link via source_to_provider when available
  CAST(NULL AS BIGINT) AS provider_id,

  -- Visit occurrence ID
  stv.visit_occurrence_id AS visit_occurrence_id,

  -- Visit detail ID
  CAST(NULL AS BIGINT) AS visit_detail_id,

  -- Device source value = material name
  COALESCE(m.MTRL_NM, m.MTRL_DESC) AS device_source_value,

  -- Source concept ID (0 since no mapping)
  CAST(0 AS INT) AS device_source_concept_id,

  -- Source system identifier
  'allscripts_scm' AS source_system

FROM _exponent._bronze_srgry_dmart.rpt_case_mtrl_use_vw mu

-- Join to material reference for device info
INNER JOIN _exponent._bronze_srgry_dmart.rpt_mtrl_vw m
  ON m.MTRL_DIM_ID = mu.MTRL_DIM_ID

-- Join to case for patient/visit linkage
INNER JOIN _exponent._bronze_srgry_dmart.rpt_case_vw c
  ON c.CASE_FCT_ID = mu.CASE_FCT_ID

-- Join to SCM visit via VisitIDCode (VST_ID_CD matches VisitIDCode)
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit v
  ON CAST(c.VST_ID_CD AS STRING) = CAST(v.VisitIDCode AS STRING)

-- Link to OMOP person via ClientGUID
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_scm',
       'cv3client',
       'GUID',
       CAST(v.ClientGUID AS STRING)
     )
  AND stp.active_flag = TRUE

-- Link to OMOP visit occurrence via visit GUID
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stv
  ON stv.visit_occurrence_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_scm',
       'cv3clientvisit',
       'GUID',
       CAST(v.GUID AS STRING)
     )
  AND stv.active_flag = TRUE

WHERE mu.MTRL_USE_FCT_ID IS NOT NULL
  AND mu.CASE_STRT_TS IS NOT NULL
  AND COALESCE(mu.MTRL_QTY_NUM, 0) > 0
  AND c.CNCLD_IND = 0  -- Exclude cancelled cases

In [ ]:
%sql
-- Preview the extracted data
-- SELECT * FROM silver_device_exposure_scm LIMIT 20;

In [ ]:
%sql
-- Validation: Check counts and data quality
-- SELECT
--   COUNT(*) AS total_records,
--   COUNT(DISTINCT person_id) AS distinct_patients,
--   COUNT(DISTINCT visit_occurrence_id) AS distinct_visits,
--   SUM(CASE WHEN device_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped_devices,
--   MIN(device_exposure_start_date) AS min_date,
--   MAX(device_exposure_start_date) AS max_date,
--   AVG(quantity) AS avg_quantity
-- FROM silver_device_exposure_scm;

In [ ]:
%sql
-- Merge into silver table
MERGE INTO _exponent.omop_silver.device_exposure AS t
USING silver_device_exposure_scm AS s
ON t.device_exposure_source_value = s.device_exposure_source_value

WHEN MATCHED AND NOT (
     t.person_id <=> s.person_id
 AND t.device_concept_id <=> s.device_concept_id
 AND t.device_exposure_start_date <=> s.device_exposure_start_date
 AND t.device_exposure_start_datetime <=> s.device_exposure_start_datetime
 AND t.device_exposure_end_date <=> s.device_exposure_end_date
 AND t.device_exposure_end_datetime <=> s.device_exposure_end_datetime
 AND t.device_type_concept_id <=> s.device_type_concept_id
 AND t.unique_device_id <=> s.unique_device_id
 AND t.quantity <=> s.quantity
 AND t.provider_id <=> s.provider_id
 AND t.visit_occurrence_id <=> s.visit_occurrence_id
 AND t.visit_detail_id <=> s.visit_detail_id
 AND t.device_source_value <=> s.device_source_value
 AND t.device_source_concept_id <=> s.device_source_concept_id
 AND t.source_system <=> s.source_system
)
THEN UPDATE SET
  t.person_id = s.person_id,
  t.device_concept_id = s.device_concept_id,
  t.device_exposure_start_date = s.device_exposure_start_date,
  t.device_exposure_start_datetime = s.device_exposure_start_datetime,
  t.device_exposure_end_date = s.device_exposure_end_date,
  t.device_exposure_end_datetime = s.device_exposure_end_datetime,
  t.device_type_concept_id = s.device_type_concept_id,
  t.unique_device_id = s.unique_device_id,
  t.quantity = s.quantity,
  t.provider_id = s.provider_id,
  t.visit_occurrence_id = s.visit_occurrence_id,
  t.visit_detail_id = s.visit_detail_id,
  t.device_source_value = s.device_source_value,
  t.device_source_concept_id = s.device_source_concept_id,
  t.source_system = s.source_system,
  t.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  device_exposure_source_value,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  source_system,
  last_mod_tsp
)
VALUES (
  s.device_exposure_source_value,
  s.person_id,
  s.device_concept_id,
  s.device_exposure_start_date,
  s.device_exposure_start_datetime,
  s.device_exposure_end_date,
  s.device_exposure_end_datetime,
  s.device_type_concept_id,
  s.unique_device_id,
  s.quantity,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.device_source_value,
  s.device_source_concept_id,
  s.source_system,
  CURRENT_TIMESTAMP()
);

In [ ]:
%sql
-- Insert into mapping table
INSERT INTO _exponent.omop_mapping.source_to_device_exposure (
  source_system,
  device_exposure_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp,
  merge_id,
  merge_reason
)
SELECT
  source_distinct.source_system,
  source_distinct.device_exposure_source_value,
  TRUE AS active_flag,
  CURRENT_TIMESTAMP() AS created_tsp,
  COALESCE(source_distinct.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
  NULL AS merge_id,
  NULL AS merge_reason
FROM (
  SELECT DISTINCT
    source_system,
    device_exposure_source_value,
    last_mod_tsp
  FROM _exponent.omop_silver.device_exposure
  WHERE source_system = 'allscripts_scm'
    AND device_exposure_source_value IS NOT NULL
) source_distinct
LEFT ANTI JOIN _exponent.omop_mapping.source_to_device_exposure existing
  ON source_distinct.device_exposure_source_value = existing.device_exposure_source_value;

In [ ]:
%sql
-- Merge into gold (OMOP) table
MERGE INTO _exponent.omop.device_exposure AS target
USING (
  SELECT
    source_to_device_exposure.device_exposure_id,
    device_exposure.device_exposure_source_value,
    device_exposure.person_id,
    device_exposure.device_concept_id,
    device_exposure.device_exposure_start_date,
    device_exposure.device_exposure_start_datetime,
    device_exposure.device_exposure_end_date,
    device_exposure.device_exposure_end_datetime,
    device_exposure.device_type_concept_id,
    device_exposure.unique_device_id,
    device_exposure.quantity,
    device_exposure.provider_id,
    device_exposure.visit_occurrence_id,
    device_exposure.visit_detail_id,
    device_exposure.device_source_value,
    device_exposure.device_source_concept_id,
    device_exposure.source_system,
    device_exposure.last_mod_tsp
  FROM _exponent.omop_silver.device_exposure
  JOIN _exponent.omop_mapping.source_to_device_exposure
    ON device_exposure.device_exposure_source_value = source_to_device_exposure.device_exposure_source_value
   AND source_to_device_exposure.active_flag = TRUE
  WHERE device_exposure.source_system = 'allscripts_scm'
) AS source
ON target.device_exposure_id = source.device_exposure_id

WHEN MATCHED AND NOT (
     target.person_id <=> source.person_id
 AND target.device_concept_id <=> source.device_concept_id
 AND target.device_exposure_start_date <=> source.device_exposure_start_date
 AND target.device_exposure_start_datetime <=> source.device_exposure_start_datetime
 AND target.device_exposure_end_date <=> source.device_exposure_end_date
 AND target.device_exposure_end_datetime <=> source.device_exposure_end_datetime
 AND target.device_type_concept_id <=> source.device_type_concept_id
 AND target.unique_device_id <=> source.unique_device_id
 AND target.quantity <=> source.quantity
 AND target.provider_id <=> source.provider_id
 AND target.visit_occurrence_id <=> source.visit_occurrence_id
 AND target.visit_detail_id <=> source.visit_detail_id
 AND target.device_source_value <=> source.device_source_value
 AND target.device_source_concept_id <=> source.device_source_concept_id
) THEN UPDATE SET
  target.person_id = source.person_id,
  target.device_concept_id = source.device_concept_id,
  target.device_exposure_start_date = source.device_exposure_start_date,
  target.device_exposure_start_datetime = source.device_exposure_start_datetime,
  target.device_exposure_end_date = source.device_exposure_end_date,
  target.device_exposure_end_datetime = source.device_exposure_end_datetime,
  target.device_type_concept_id = source.device_type_concept_id,
  target.unique_device_id = source.unique_device_id,
  target.quantity = source.quantity,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.device_source_value = source.device_source_value,
  target.device_source_concept_id = source.device_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  device_exposure_id,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id
) VALUES (
  source.device_exposure_id,
  source.person_id,
  source.device_concept_id,
  source.device_exposure_start_date,
  source.device_exposure_start_datetime,
  source.device_exposure_end_date,
  source.device_exposure_end_datetime,
  source.device_type_concept_id,
  source.unique_device_id,
  source.quantity,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.device_source_value,
  source.device_source_concept_id
);

In [ ]:
%sql
-- Verify: Check layer counts
-- SELECT 'Silver' AS layer, COUNT(*) AS record_count 
-- FROM _exponent.omop_silver.device_exposure 
-- WHERE source_system = 'allscripts_scm'
-- UNION ALL
-- SELECT 'Mapping' AS layer, COUNT(*) AS record_count 
-- FROM _exponent.omop_mapping.source_to_device_exposure 
-- WHERE source_system = 'allscripts_scm'
-- UNION ALL
-- SELECT 'Gold' AS layer, COUNT(*) AS record_count 
-- FROM _exponent.omop.device_exposure;

In [ ]:
%sql
-- Sample gold records
-- SELECT * FROM _exponent.omop.device_exposure
-- WHERE device_exposure_id IN (
--   SELECT device_exposure_id 
--   FROM _exponent.omop_mapping.source_to_device_exposure
--   WHERE source_system = 'allscripts_scm'
-- )
-- LIMIT 20;